# SME Finance Copilot — Qwen3-4B-Instruct-2507 QLoRA Fine-Tuning

This notebook fine-tunes **Qwen3-4B-Instruct-2507** on the institutional SME fractional CFO reasoning dataset.

### Kaggle Setup Instructions:
1. Under **Settings** (right sidebar), ensure **Accelerator** is set to **GPU T4** (or GPU T4 x2).
2. Ensure **Internet** is toggled **ON**.
3. Upload `dataset_qwen3_ready.zip` via **Add Input -> Upload Data** (or place the files in `/kaggle/working`).
4. Click **Run All**.
5. When finished, download `sme_finance_lora_qwen3_4b.zip` from the **Output** panel.

In [ ]:
# 1. Remove torchaudio to prevent Kaggle CUDA mismatch, then install deps
!pip uninstall -y torchaudio
!pip install -q transformers peft bitsandbytes datasets trl accelerate


In [ ]:
# 2. Verify GPU status
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    raise RuntimeError('Please enable GPU accelerator in Kaggle notebook settings!')

In [ ]:
# 3. Unzip dataset if uploaded as zip
import os, glob, zipfile

zip_files = glob.glob('/kaggle/input/**/dataset_qwen3_ready.zip', recursive=True) + glob.glob('./dataset_qwen3_ready.zip')
if zip_files:
    print(f'Unpacking {zip_files[0]}...')
    with zipfile.ZipFile(zip_files[0], 'r') as z:
        z.extractall('/kaggle/working/dataset')
    print('Dataset extracted successfully.')
else:
    print('Checking for unzipped jsonl files...')

In [ ]:
# 4. Run the QLoRA Fine-Tuning Pipeline (TURBO: ~25 min on Tesla T4)
import os, glob, zipfile, torch, sys, types

# 1. Mock torchaudio to prevent Kaggle audio conflicts
sys.modules['torchaudio'] = types.ModuleType('torchaudio')
try:
    import transformers.utils.import_utils as t_utils
    t_utils.is_torchaudio_available = lambda: False
    t_utils._torchaudio_available = False
except Exception:
    pass

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
try:
    from trl import SFTConfig, SFTTrainer
    HAS_SFT_CONFIG = True
except ImportError:
    from trl import SFTTrainer
    HAS_SFT_CONFIG = False

MODEL_ID = 'Qwen/Qwen3-4B-Instruct-2507'
OUTPUT_DIR = '/kaggle/working/sme-finance-lora-qwen3-4b'
ZIP_OUTPUT = '/kaggle/working/sme_finance_lora_qwen3_4b.zip'

# 2. Locate training files
train_candidates = glob.glob('/kaggle/input/**/finetune_train.jsonl', recursive=True) + glob.glob('/kaggle/working/**/finetune_train.jsonl', recursive=True) + ['./finetune_train.jsonl']
val_candidates = glob.glob('/kaggle/input/**/finetune_val.jsonl', recursive=True) + glob.glob('/kaggle/working/**/finetune_val.jsonl', recursive=True) + ['./finetune_val.jsonl']

assert train_candidates, 'finetune_train.jsonl not found! Please attach dataset_qwen3_ready.zip'
train_path = train_candidates[0]
val_path = val_candidates[0] if val_candidates else None
print(f'Train path: {train_path}')
print(f'Val path:   {val_path}')

# 3. Hardware precision: Tesla T4 (CC 7.5) requires fp16
is_ampere_or_newer = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
compute_dtype = torch.bfloat16 if is_ampere_or_newer else torch.float16
print(f'GPU Compute Capability: {torch.cuda.get_device_capability() if torch.cuda.is_available() else "None"} -> Using dtype: {compute_dtype}')

# 4. 4-bit NormalFloat4 config in FP16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print('Loading tokenizer and 4-bit base model in FP16...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=compute_dtype,
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

# 5. LoRA configuration
target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Eliminate BFloat16 on Tesla T4: cast base tensors to float16, and all trainable adapter tensors to float32
for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

# 6. Load & format dataset
data_files = {'train': train_path}
if val_path:
    data_files['val'] = val_path
raw_datasets = load_dataset('json', data_files=data_files)

def format_chatml(batch):
    formatted = []
    for conversation in batch['messages']:
        text = tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=False,
        )
        formatted.append(text)
    return {'text': formatted}

train_dataset = raw_datasets['train'].map(format_chatml, batched=True, load_from_cache_file=False)
val_dataset = raw_datasets['val'].map(format_chatml, batched=True, load_from_cache_file=False) if val_path else None

# 7. Fast Turbo Training Configuration (~25 min on Tesla T4)
training_kwargs = {
    'output_dir': OUTPUT_DIR,
    'num_train_epochs': 1,                 # 1 epoch (193 steps) is optimal for LoRA instruction alignment
    'per_device_train_batch_size': 4,      # 4 batch size fits in 16GB T4 (~8.2 GB utilized)
    'gradient_accumulation_steps': 4,       # Effective batch size = 16
    'learning_rate': 2e-4,
    'lr_scheduler_type': 'cosine',
    'warmup_steps': 15,
    'logging_steps': 5,
    'save_strategy': 'epoch',
    'save_total_limit': 1,
    'fp16': not is_ampere_or_newer,
    'bf16': is_ampere_or_newer,
    'gradient_checkpointing': False,       # False eliminates forward-recomputation (1.8x speedup)
    'optim': 'paged_adamw_8bit',
    'report_to': 'none',
    'max_length': 1536,                    # 1536 covers 95%+ of examples with 1.8x faster attention
    'dataset_text_field': 'text',
    'eval_strategy': 'no',                 # Skip mid-training eval to avoid pauses
}

if HAS_SFT_CONFIG:
    sft_config = SFTConfig(**training_kwargs)
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        args=sft_config,
    )
else:
    from transformers import TrainingArguments
    training_kwargs.pop('max_length', None)
    training_kwargs.pop('dataset_text_field', None)
    training_kwargs.pop('eval_strategy', None)
    if val_dataset:
        training_kwargs['evaluation_strategy'] = 'epoch'
    args = TrainingArguments(**training_kwargs)
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        dataset_text_field='text',
        max_seq_length=1536,
        tokenizer=tokenizer,
        args=args,
    )

print('Starting fine-tuning...')
trainer.train()

# 8. Save LoRA weights & create download zip
print(f'Saving LoRA weights to {OUTPUT_DIR}...')
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f'Creating downloadable zip: {ZIP_OUTPUT}...')
with zipfile.ZipFile(ZIP_OUTPUT, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(OUTPUT_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, OUTPUT_DIR)
            zipf.write(full_path, rel_path)

print('DONE! Download sme_finance_lora_qwen3_4b.zip from the Output panel.')



In [ ]:
# 5. Quick Test of the Fine-Tuned Model
from peft import AutoPeftModelForCausalLM

print('Testing inference with trained adapter...')
test_prompt = (
    '<|im_start|>system\n'
    'You are the Autonomous SME Finance Copilot fractional CFO.\n'
    'Verified Financial Engine Outputs:\n'
    '- Cash Balance: $65,000.00\n'
    '- Average Daily Burn: $1,250.00/day\n'
    '- Cash Runway: 52 days\n\n'
    'Executive Advisory Guidelines:\n'
    'State the runway and burn directly. Never invent figures.<|im_end|>\n'
    '<|im_start|>user\n'
    'What is our current cash runway?<|im_end|>\n'
    '<|im_start|>assistant\n'
)
inputs = tokenizer(test_prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=150, temperature=0.2, do_sample=False)
print(tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))
# Display clickable download link
from IPython.display import FileLink
display(FileLink('sme_finance_lora_qwen3_4b.zip'))
